## Auto Loader Options Without `cloudFiles` Prefix

Auto Loader options fall into two categories:

**1. Auto Loader-Specific Options** (REQUIRE `cloudFiles` prefix):
* `cloudFiles.format`
* `cloudFiles.schemaLocation`
* `cloudFiles.schemaEvolutionMode`
* `cloudFiles.inferColumnTypes`
* `cloudFiles.schemaHints`
* `cloudFiles.useNotifications`
* All other Auto Loader-specific configurations

**2. Standard Spark DataFrameReader Options** (NO `cloudFiles` prefix needed):

### Generic Options (All Formats)
* `mode` - Parser mode: `PERMISSIVE`, `DROPMALFORMED`, `FAILFAST`
* `badRecordsPath` - Path to store bad records
* `encoding` or `charset` - Character encoding
* `dateFormat` - Date parsing format
* `timestampFormat` - Timestamp parsing format
* `timeZone` - Timezone for parsing
* `rescuedDataColumn` - Name for rescued data column
* `readerCaseSensitive` - Case sensitivity behavior

### CSV-Specific Options
* `header` - Whether first row is header
* `delimiter` or `sep` - Field separator character
* `quote` - Quote character
* `escape` - Escape character
* `multiLine` - Whether records span multiple lines
* `comment` - Line comment character
* `nullValue` - String representation of null
* `emptyValue` - String representation of empty value
* `enforceSchema` - Force schema application
* `skipRows` - Number of rows to skip

### JSON-Specific Options
* `multiLine` - Whether JSON spans multiple lines
* `primitivesAsString` - Infer primitives as strings
* `allowComments` - Allow C/Java style comments
* `allowUnquotedFieldNames` - Allow unquoted field names
* `allowSingleQuotes` - Allow single quotes
* `allowNumericLeadingZeros` - Allow leading zeros
* `allowBackslashEscapingAnyCharacter` - Escape behavior
* `dropFieldIfAllNull` - Ignore all-null columns
* `prefersDecimal` - Prefer decimal over float/double
* `lineSep` - Line separator string

### Parquet/Avro/ORC Options
* `mergeSchema` - Merge schemas across files

---

## Example With Mixed Options

```python
df = (spark.readStream
  .format("cloudFiles")
  # Auto Loader options (need cloudFiles prefix)
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", "/path/to/schema")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.inferColumnTypes", "true")
  
  # Standard Spark CSV options (NO cloudFiles prefix)
  .option("header", "true")
  .option("delimiter", "|")
  .option("quote", "\"")
  .option("multiLine", "false")
  .option("mode", "PERMISSIVE")
  .option("nullValue", "NULL")
  
  .load("/path/to/source")
)
```

**Key Rule**: Auto Loader features use `cloudFiles.` prefix; standard file format parsing options don't.

## Autoloader with SchemaEvolution Options

### Basic Auto Loader with Schema Evolution (Default Mode)

```python
# Reads JSON files with automatic schema inference and evolution
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/path/to/schema/location")
  .load("/path/to/source/data")
)

# Write to Delta table
(df.writeStream
  .option("checkpointLocation", "/path/to/checkpoint")
  .option("mergeSchema", "true")  # Allow schema evolution in target table
  .toTable("my_catalog.my_schema.my_table")
)
```

### Rescue Mode (Never Fails on Schema Changes)

```python
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/path/to/schema/location")
  .option("cloudFiles.schemaEvolutionMode", "rescue")
  .option("rescuedDataColumn", "_rescued_data")  # Optional: rename rescued column
  .load("/path/to/source/data")
)

(df.writeStream
  .option("checkpointLocation", "/path/to/checkpoint")
  .toTable("my_catalog.my_schema.my_table")
)
```

### Type Widening Mode (Auto-widens compatible types)

```python
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "parquet")
  .option("cloudFiles.schemaLocation", "/path/to/schema/location")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening")
  .load("/path/to/source/data")
)

(df.writeStream
  .option("checkpointLocation", "/path/to/checkpoint")
  .toTable("my_catalog.my_schema.my_table")
)
```

### With Schema Hints (Control specific column types)

```python
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/path/to/schema/location")
  .option("cloudFiles.inferColumnTypes", "true")  # Infer types instead of all strings
  .option("cloudFiles.schemaHints", "user_id INT, amount DECIMAL(10,2), created_at TIMESTAMP")
  .load("/path/to/source/data")
)

(df.writeStream
  .option("checkpointLocation", "/path/to/checkpoint")
  .toTable("my_catalog.my_schema.my_table")
)
```

### CSV with Schema Evolution

```python
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", "/path/to/schema/location")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .option("cloudFiles.inferColumnTypes", "true")
  # Standard CSV options (no cloudFiles prefix)
  .option("header", "true")
  .option("delimiter", ",")
  .option("mode", "PERMISSIVE")
  .load("/path/to/source/data")
)

(df.writeStream
  .option("checkpointLocation", "/path/to/checkpoint")
  .option("mergeSchema", "true")
  .toTable("my_catalog.my_schema.my_table")
)
```

### SQL Syntax (Lakeflow Pipelines)

```sql
CREATE OR REFRESH STREAMING TABLE my_target_table AS
SELECT * FROM cloud_files(
  "/path/to/source/data",
  "json",
  map(
    "cloudFiles.schemaLocation", "/path/to/schema/location",
    "cloudFiles.schemaEvolutionMode", "addNewColumns",
    "cloudFiles.inferColumnTypes", "true"
  )
)
```

### Key Schema Evolution Modes

| Mode | Behavior |
|------|----------|
| `addNewColumns` (default) | Stream fails temporarily, adds new columns, restarts automatically |
| `rescue` | Never evolves schema, new columns go to `_rescued_data` |
| `failOnNewColumns` | Stream fails permanently until schema is fixed |
| `none` | Ignores new columns, doesn't fail |
| `addNewColumnsWithTypeWidening` | Adds columns + auto-widens types (int→long, float→double) |

### Important Options

* `cloudFiles.schemaLocation` - **Required** for schema evolution
* `cloudFiles.schemaEvolutionMode` - Controls how new columns are handled
* `cloudFiles.inferColumnTypes` - Set to `true` to infer types (default: strings for JSON/CSV)
* `rescuedDataColumn` - Column name for unexpected/mismatched data
* `cloudFiles.schemaHints` - Override inferred types for specific columns